# Libraries

In [ ]:
# =========================================================
# Standard Library
# =========================================================
import copy
import gc
import itertools
import math
import os
import random
import time

# =========================================================
# Core Data / Math Libraries
# =========================================================
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial.distance import jensenshannon

# =========================================================
# PyTorch
# =========================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

# =========================================================
# Scikit-learn
# =========================================================
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    mean_squared_log_error,
    precision_score,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# =========================================================
# Gradient Boosting Libraries
# =========================================================
import lightgbm as lgb
import xgboost as xgb

# =========================================================
# Kaggle Input Files
# =========================================================
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Main Functions

In [ ]:

# =========================================================
# Utilities
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def reduce_mem_usage(df):
    for col in df.columns:
        col_type = df[col].dtype

        if pd.api.types.is_object_dtype(col_type) or pd.api.types.is_datetime64_any_dtype(col_type):
            continue

        if pd.api.types.is_integer_dtype(col_type):
            c_min = df[col].min()
            c_max = df[col].max()

            if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.int64)

        elif pd.api.types.is_float_dtype(col_type):
            df[col] = df[col].astype(np.float32)

    return df


def features_engineering(df):
    df.loc[:, 'hour'] = df['timestamp'].dt.hour
    df.loc[:, 'day'] = df['timestamp'].dt.day
    df.loc[:, 'month'] = df['timestamp'].dt.month
    df.loc[:, 'weekday'] = df['timestamp'].dt.weekday
    df['square_feet'] = np.log1p(df['square_feet'].astype(np.float32))
    df.drop(
        ['sea_level_pressure', 'wind_direction', 'wind_speed', 'year_built', 'floor_count'],
        axis=1,
        inplace=True
    )
    return df


def normalize_train_features(train_features, feature_cols, cat_threshold=5):
    binary_cols = []
    categorical_cols = []
    numeric_cols = []

    for col in feature_cols:
        unique_vals = train_features[col].dropna().unique()

        if set(unique_vals).issubset({0, 1}):
            binary_cols.append(col)
        elif np.issubdtype(train_features[col].dtype, np.integer) and len(unique_vals) <= cat_threshold:
            categorical_cols.append(col)
        else:
            numeric_cols.append(col)

    print("Numeric:", numeric_cols)
    print("Binary:", binary_cols)
    print("Categorical:", categorical_cols)

    scaler = StandardScaler()
    train_features[numeric_cols] = scaler.fit_transform(train_features[numeric_cols])

    return train_features, scaler, numeric_cols


# =========================================================
# Data Splitting
# =========================================================
def split_each_building_by_month(
    df,
    building_col,
    time_col,
    train_months=None,
    valid_months=None,
    test_months=None
):
    train_parts, valid_parts, test_parts = [], [], []

    for bld, g in df.groupby(building_col, sort=False):
        g = g.sort_values(time_col).reset_index(drop=True)
        months = g[time_col].dt.month

        if train_months is not None:
            train_parts.append(g[months.isin(train_months)])

        if valid_months is not None:
            valid_parts.append(g[months.isin(valid_months)])

        if test_months is not None:
            test_parts.append(g[months.isin(test_months)])

    train_df = pd.concat(train_parts, axis=0).reset_index(drop=True) if train_parts else pd.DataFrame()
    valid_df = pd.concat(valid_parts, axis=0).reset_index(drop=True) if valid_parts else pd.DataFrame()
    test_df = pd.concat(test_parts, axis=0).reset_index(drop=True) if test_parts else pd.DataFrame()

    return train_df, valid_df, test_df


def split_each_building(df, building_col, time_col, train_ratio=0.1, valid_ratio=0.10):
    train_parts, valid_parts, test_parts = [], [], []

    for bld, g in df.groupby(building_col, sort=False):
        g = g.sort_values(time_col).reset_index(drop=True)

        n = len(g)
        n_train = int(n * train_ratio)
        n_valid = int(n * valid_ratio)

        train_parts.append(g.iloc[:n_train])
        valid_parts.append(g.iloc[n_train:n_train + n_valid])
        test_parts.append(g.iloc[n_train + n_valid:])

    train_df = pd.concat(train_parts, axis=0).reset_index(drop=True)
    valid_df = pd.concat(valid_parts, axis=0).reset_index(drop=True)
    test_df = pd.concat(test_parts, axis=0).reset_index(drop=True)

    return train_df, valid_df, test_df


# =========================================================
# Sequence Creation
# =========================================================
def make_sequences_by_building(
    df_part,
    feature_cols,
    target_col,
    building_col,
    building_idx_col,
    seq_len=48,
    horizon=0
):
    X_seq, y_seq, b_seq = [], [], []

    for bld, g in df_part.groupby(building_col, sort=False):
        X_b = g[feature_cols].values.astype(np.float32)
        y_b = g[target_col].values.astype(np.float32)
        b_idx = int(g[building_idx_col].iloc[0])

        max_start = len(X_b) - seq_len - horizon + 1
        if max_start <= 0:
            continue

        for s in range(max_start):
            X_seq.append(X_b[s:s + seq_len])

            # predict only final target of the window
            target_idx = s + seq_len - 1 + horizon
            y_seq.append(y_b[target_idx])

            b_seq.append(b_idx)

    X_seq = np.asarray(X_seq, dtype=np.float32)
    y_seq = np.asarray(y_seq, dtype=np.float32).reshape(-1, 1)
    b_seq = np.asarray(b_seq, dtype=np.int64)

    return X_seq, y_seq, b_seq


class PrebuiltSequenceDataset(Dataset):
    def __init__(self, X, y, building_ids):
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32)
        building_ids = np.asarray(building_ids, dtype=np.int64)

        if y.ndim == 1:
            y = y[:, None]

        self.X = X
        self.y = y
        self.building_ids = building_ids

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        # ✅ Just index (fast)
        return self.X[idx], self.y[idx], self.building_ids[idx]

from numpy.ma import tanh

LOG_2PI = math.log(2.0 * math.pi)

# =========================================================
# Probability / Loss Functions
# =========================================================
def gaussian_nll(y_true, mu, logvar):
    var = torch.exp(logvar).clamp_min(1e-6)
    return 0.5 * ((y_true - mu) ** 2 * torch.exp(-logvar) + logvar + LOG_2PI)


def standard_normal_nll(z):
    return 0.5 * (z ** 2 + LOG_2PI)



# =========================================================
# EM Step Functions
# =========================================================
@torch.no_grad()
def e_step_sample_and_weight(model, obs, y, building_ids, n_particles=5):
    particle_x = []
    log_weights = []

    for _ in range(n_particles):
        out = model.forward_sample(obs, building_ids)

        x_seq = out["x_seq"]          # [B, T, D]
        x_mu = out["x_mu"]            # [B, T, D]
        x_logvar = out["x_logvar"]    # [B, T, D]
        y_mu = out["y_mu"]            # [B, 1]
        y_logvar = out["y_logvar"]    # [B, 1]

        log_py = -gaussian_nll(y, y_mu, y_logvar).sum(dim=1)   # [B]

        log_px = -gaussian_nll(
            x_seq[:, -1, :],
            x_mu[:, -1, :],
            x_logvar[:, -1, :]
        ).sum(dim=1)   # [B]

        lw = log_py

        particle_x.append(x_seq)
        log_weights.append(lw)

    particle_x = torch.stack(particle_x, dim=0)   # [N, B, T, D]
    log_weights = torch.stack(log_weights, dim=0) # [N, B]

    weights = torch.softmax(log_weights, dim=0)

    N, B = weights.shape
    resampled_idx = []
    for b in range(B):
        idx_b = torch.multinomial(weights[:, b], num_samples=N, replacement=True)
        resampled_idx.append(idx_b)
    resampled_idx = torch.stack(resampled_idx, dim=1)   # [N, B]

    resampled_x = []
    batch_idx = torch.arange(B, device=obs.device)

    for i in range(N):
        idx_i = resampled_idx[i]
        chosen_x = particle_x[idx_i, batch_idx]         # [B, T, D]
        resampled_x.append(chosen_x)

    resampled_x = torch.stack(resampled_x, dim=0)       # [N, B, T, D]
    return resampled_x


def m_step_loss(model, obs, y, building_ids, resampled_x, lambda_x=1.0):
    N = resampled_x.shape[0]

    total_y_nll = 0.0
    total_x_nll = 0.0

    x0, x_mu, x_logvar = model.encode(obs, building_ids)


    for i in range(N):
        x_seq_i = resampled_x[i]
        y_mu_i, y_logvar_i = model.decode_from_x(obs, x0, x_seq_i)

        # end-of-window output loss only
        y_nll_i = gaussian_nll(y, y_mu_i, y_logvar_i).mean()

        # final latent-state consistency only
        x_nll_i = gaussian_nll(
            x_seq_i[:, -1, :],
            x_mu[:, -1, :],
            x_logvar[:, -1, :]
        ).mean()

        total_y_nll += y_nll_i
        total_x_nll += x_nll_i

    total_y_nll /= N
    total_x_nll /= N
    mu_penalty = x_mu.pow(2).mean()
    loss = total_y_nll + total_x_nll + 0.001 * mu_penalty

    return loss, total_y_nll.detach(), total_x_nll.detach()


def train_one_epoch_em(model, loader, optimizer):
    model.train()
    total_loss = 0.0

    for obs, y, building_ids in loader:
        obs = obs.to(device)
        y = y.to(device)  # [B, 1]
        building_ids = building_ids.to(device)

        # E-step
        model.eval()
        with torch.no_grad():
            resampled_x = e_step_sample_and_weight(
                model, obs, y, building_ids,
                n_particles=N_PARTICLES
            )

        # M-step
        model.train()
        optimizer.zero_grad()

        loss, y_loss, x_loss = m_step_loss(
            model, obs, y, building_ids, resampled_x
        )

        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += float(loss.item())

    return total_loss / len(loader)


@torch.no_grad()
def valid_one_epoch_em(model, loader):
    model.eval()
    total_loss = 0.0
    total_y = 0.0
    total_x = 0.0

    for obs, y, building_ids in loader:
        obs = obs.to(device)
        y = y.to(device)
        building_ids = building_ids.to(device)

        resampled_x = e_step_sample_and_weight(
            model, obs, y, building_ids,
            n_particles=N_PARTICLES
        )

        loss, y_loss, x_loss= m_step_loss(
            model, obs, y, building_ids, resampled_x
        )

        total_loss += float(loss.item())
        total_y += float(y_loss.item())
        total_x += float(x_loss.item())

    return total_loss / len(loader)


# =========================================================
# Evaluation
# =========================================================
def Model_Evaluate(model, X_valid, y_valid, b_valid, model_name, benchmark):
    valid_preds = model.predict(X_valid, b_valid)

    y_valid = np.asarray(y_valid).reshape(-1)
    valid_preds = np.asarray(valid_preds).reshape(-1)

    y_valid_expm1 = np.expm1(y_valid)
    valid_preds_expm1 = np.expm1(valid_preds)

    valid_rmse = np.sqrt(mean_squared_error(y_valid_expm1, valid_preds_expm1))
    valid_log_rmse = np.sqrt(mean_squared_error(y_valid, valid_preds))

    valid_r2 = r2_score(y_valid_expm1, valid_preds_expm1)
    valid_mad = np.mean(np.abs(y_valid_expm1 - valid_preds_expm1))

    denom = np.where(np.abs(y_valid_expm1) < 1e-8, 1e-8, np.abs(y_valid_expm1))
    valid_mape = np.median(np.abs((y_valid_expm1 - valid_preds_expm1) / denom)) * 100

    valid_wape = (
        np.sum(np.abs(y_valid_expm1 - valid_preds_expm1))
        / max(np.sum(np.abs(y_valid_expm1)), 1e-8)
        * 100
    )

    print(f"\nModel: {model_name}")
    print(f"Validation Log RMSE: {valid_log_rmse:.4f}")
    print(f"Validation RMSE: {valid_rmse:.4f}")
    print(f"Validation R²: {valid_r2:.4f}")
    print(f"Validation MAD: {valid_mad:.4f}")
    print(f"Validation MAPE: {valid_mape:.4f}")
    print(f"Validation WAPE: {valid_wape:.4f}%")

    return (
        y_valid_expm1.tolist(),
        valid_preds_expm1,
        valid_log_rmse,
        valid_rmse,
        valid_r2,
        valid_mad,
        valid_mape,
        valid_wape)




def compute_rmse(model, X_valid, y_valid, b_valid):
    preds = model.predict(X_valid, b_valid)

    y_valid = np.asarray(y_valid).reshape(-1)
    preds = np.asarray(preds).reshape(-1)

    logrmse = np.sqrt(np.mean((y_valid - preds) ** 2))

    y_valid = np.expm1(y_valid)
    preds = np.expm1(preds)

    rmse = np.sqrt(np.mean((y_valid - preds) ** 2))

    return rmse, logrmse


@torch.no_grad()
def mc_predict_batch(model, obs, building_ids, mc_samples=30):
    """
    Returns:
        pred_mean: [B, 1]
        pred_var : [B, 1]
    """
    model.eval()

    mus = []
    vars_ = []

    for _ in range(mc_samples):
        out = model.forward_sample(obs, building_ids)
        mus.append(out["y_mu"])                   # [B, 1]
        vars_.append(torch.exp(out["y_logvar"])) # [B, 1]

    mus = torch.stack(mus, dim=0)      # [S, B, 1]
    vars_ = torch.stack(vars_, dim=0)  # [S, B, 1]

    pred_mean = mus.mean(dim=0)
    epistemic_var = mus.var(dim=0, unbiased=False)
    aleatoric_var = vars_.mean(dim=0)
    pred_var = (epistemic_var + aleatoric_var).clamp_min(1e-8)

    return pred_mean, pred_var

# Dataset Loading and Filtering for Manufacturing

Mounted at /content/drive


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# # Define data path (make sure your dataset is in this path)
# DATA_PATH = '/content/drive/MyDrive/SEG1PublicData'

###### The dataset in available online in Kaggle
# Original Data Source: https://www.kaggle.com/competitions/ashrae-energy-prediction/data
# Main files needed are (building_metadata.csv,sample_submission.csv, test.csv, train.csv, weather_test.csv, weather_train.csv)
# Load the Dataset from your saved folder
# some data preperation/cleaning steps are from https://www.kaggle.com/code/rebinos/energy-consumption-prediction-xgb-lgbm-and-rf/notebook


# Load the dataset
train_df = pd.read_csv(DATA_PATH+ '/train.csv')
# Remove outliers
train_df = train_df [ train_df['building_id'] != 1099 ]
train_df = train_df.query('not (building_id <= 104 & meter == 0 & timestamp <= "2016-05-20")')

## PReprocessing the data
building_df = pd.read_csv(DATA_PATH + '/building_metadata.csv')
weather_train_df = pd.read_csv(DATA_PATH + '/weather_train.csv')

## Convert timestamp to datetime
train_df['timestamp'] = pd.to_datetime(train_df['timestamp'])
weather_train_df['timestamp'] = pd.to_datetime(weather_train_df['timestamp'])


train_df = reduce_mem_usage(train_df)
weather_train_df = reduce_mem_usage(weather_train_df)
building_df = reduce_mem_usage(building_df)


# Merging building metadata with train and test data
train_df = train_df.merge(building_df, on='building_id', how='left')
 # only electricity meters
train_df=train_df[train_df.meter==0]

# Merge with weather data only if 'site_id' exists
if 'site_id' in train_df.columns and 'site_id' in weather_train_df.columns:
    train_df = train_df.merge(weather_train_df, on=['site_id', 'timestamp'], how='left')

train_df = train_df[train_df['primary_use'].isin(['Manufacturing/industrial', 'Warehouse/storage'])]

# Label encode 'primary_use' only if it exists in both datasets
if 'primary_use' in train_df.columns:
    le = LabelEncoder()
    train_df['primary_use'] = le.fit_transform(train_df['primary_use'])

# Apply feature engineering
train_df = features_engineering(train_df)

In [ ]:
train_df

# Missing Imputation

In [ ]:

# Keep building_id in both train_features and test_features
train_features = train_df.drop(columns=['meter' ]).copy()

# numeric columns to fill, excluding building_id itself
numeric_cols = train_features.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'building_id']
numeric_cols = [c for c in numeric_cols if c != 'timestamp']
numeric_cols = [c for c in numeric_cols if c != 'meter_reading']


# global fallback medians from train only
global_medians = train_features[numeric_cols].median()

# per-building medians from train only
building_medians = train_features.groupby('building_id')[numeric_cols].median()

# fill train
for col in numeric_cols:
    train_features[col] = (
        train_features[col]
        .fillna(train_features['building_id'].map(building_medians[col]))
        .fillna(global_medians[col])
    )


# optional: reduce memory
for col in numeric_cols:
    train_features[col] = train_features[col].astype(np.float32)


gc.collect()

90

# New features, Normalization, and Finalizing Dataset

In [ ]:
train_features['meter_reading'] = np.log1p(train_features['meter_reading'])
# Cyclical encoding
train_features['hour_sin']   = np.sin(2 * np.pi * train_features['hour'] / 24)
train_features['hour_cos']   = np.cos(2 * np.pi * train_features['hour'] / 24)
train_features['dow_sin']    = np.sin(2 * np.pi * train_features['weekday'] / 7)
train_features['dow_cos']    = np.cos(2 * np.pi * train_features['weekday'] / 7)
train_features['month_sin']  = np.sin(2 * np.pi * train_features['month'] / 12)
train_features['month_cos']  = np.cos(2 * np.pi * train_features['month'] / 12)
# Optional binary
train_features['is_weekend'] = (train_features['weekday'] >= 5).astype(int)

# ====== YOU SET THESE ======
feature_cols = ['site_id', 'primary_use', 'square_feet',
       'air_temperature', 'cloud_coverage', 'dew_temperature',
       'precip_depth_1_hr','hour_sin', 'hour_cos','dow_sin','dow_cos','month_sin','month_cos','is_weekend'  ,  'weekday']


target_col = "meter_reading"
building_col = "building_id"
time_col = "timestamp"
unique_buildings = sorted(train_features[building_col].unique())
building_to_idx = {b: i for i, b in enumerate(unique_buildings)}

train_features["building_idx"] = train_features[building_col].map(building_to_idx)
num_buildings = len(train_features.building_idx.unique())

train_features = train_features.copy()
train_features[time_col] = pd.to_datetime(train_features[time_col])
train_features = train_features.sort_values([building_col, time_col]).reset_index(drop=True)

train_featuresNormalized, scaler, numeric_cols = normalize_train_features(
    train_features,
    feature_cols
)



Numeric: ['site_id', 'square_feet', 'air_temperature', 'cloud_coverage', 'dew_temperature', 'precip_depth_1_hr', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'weekday']
Binary: ['primary_use', 'is_weekend']
Categorical: []


# Main Model Strcuture

In [ ]:

# =========================================================
# Model
# =========================================================
class LatentStateEnergyModel(nn.Module):
    def __init__(
        self,
        obs_dim,
        num_buildings,
        latent_dim=16,
        enc_hidden=64,
        dec_hidden=64,
        enc_out_hidden=64,
        dec_out_hidden=64,
        enc_layers=2,
        dec_layers=2,
        dropout=0.1,
        y_dim=1
    ):
        super().__init__()

        self.x0_net = nn.Embedding(num_buildings, latent_dim)
        self.latent_dim = latent_dim

        self.encoder = nn.LSTM(
            input_size=obs_dim + latent_dim,
            hidden_size=enc_hidden,
            num_layers=enc_layers,
            batch_first=True,
            dropout=dropout if enc_layers > 1 else 0.0
        )

        self.enc_mu = nn.Sequential(
            nn.Linear(enc_hidden,enc_out_hidden),
            nn.ReLU(),
            nn.Linear(enc_out_hidden, latent_dim),

        )

        self.enc_logvar = nn.Sequential(
            nn.Linear(enc_hidden , enc_out_hidden),
            nn.ReLU(),
            nn.Linear(enc_out_hidden, latent_dim)
        )


        self.decoder = nn.LSTM(
            input_size=obs_dim,
            hidden_size=dec_hidden,
            num_layers=dec_layers,
            batch_first=True,
            dropout=dropout if dec_layers > 1 else 0.0
        )

        self.dec_mu = nn.Sequential(
            nn.Linear(dec_hidden +obs_dim +2*latent_dim, dec_out_hidden),
            nn.ReLU(),
            nn.Linear(dec_out_hidden, y_dim)
        )

        self.dec_logvar = nn.Sequential(
            nn.Linear(dec_hidden+obs_dim +2*latent_dim, dec_out_hidden),
            nn.ReLU(),
            nn.Linear(dec_out_hidden, y_dim)
        )

    def sample_x(self, mu, logvar):
        std = torch.exp(0.5 * logvar).clamp_min(1e-6)

        if torch.isnan(std).any() or torch.isinf(std).any():
          print("NaN/Inf in std")
          print(
              f"[sample_x] mu mean={mu.mean().item():.4f}, mu std={mu.std().item():.4f}, "
              f"logvar min={logvar.min().item():.4f}, logvar max={logvar.max().item():.4f}, "
              f"std mean={std.mean().item():.4f}, std max={std.max().item():.4f}"
          )
        eps = torch.randn_like(std)
        return mu + eps * std

    def encode(self, obs, building_ids):
        B, T, _ = obs.shape

        x0=torch.sigmoid(self.x0_net(building_ids))

        x0_rep = x0.unsqueeze(1).expand(-1, T, -1)

        enc_in = torch.cat([obs, x0_rep], dim=-1)

        enc_h, _ = self.encoder(enc_in)

        x_mu = 1* self.enc_mu(enc_h)
        x_logvar = self.enc_logvar(enc_h).clamp(-6.0, -3)

        return x0, x_mu, x_logvar

    def decode_from_x(self, obs, x0, x_seq):
        B, T, _ = obs.shape
        x_last = x_seq[:, -1, :]                       # [B, latent_dim]
        x_last_rep = x_last.unsqueeze(1).expand(-1, T, -1)   # [B, T, latent_dim]
        x0_rep = x0.unsqueeze(1).expand(-1, T, -1)

        # decoder sees only observations
        dec_in = torch.cat([obs], dim=-1)
        dec_h, _ = self.decoder(dec_in)

        dec_last = dec_h[:, -1, :]      # [B, dec_hidden]
        x_last = x_seq[:, -1, :]        # [B, latent_dim]
        o_last = obs[:, -1, :]#.unsqueeze(1).expand(-1, T, -1)

        head_in = torch.cat([dec_last, o_last, x_last, x0], dim=-1)

        y_mu = self.dec_mu(head_in)                  # [B, 1]
        y_logvar = self.dec_logvar(head_in).clamp(-6.0, 1)

        return y_mu, y_logvar

    def forward_sample(self, obs, building_ids):
        x0, x_mu, x_logvar = self.encode(obs, building_ids)
        x_seq = self.sample_x(x_mu, x_logvar)
        y_mu, y_logvar = self.decode_from_x(obs, x0, x_seq)

        return {
            "x0": x0,
            "x_mu": x_mu,
            "x_logvar": x_logvar,
            "x_seq": x_seq,
            "y_mu": y_mu,
            "y_logvar": y_logvar,
        }


# Algorithm Strcuture

In [ ]:
class EMSeq2SeqRegressor(BaseEstimator, RegressorMixin):

    def __init__(
        self,
        return_sequence=False,
        latent_dim=16,
        enc_hidden=64,
        dec_hidden=64,
        enc_out_hidden=64,
        dec_out_hidden=64,
        enc_layers=2,
        dec_layers=2,
        dropout=0.1,
        lr=1e-3,
        weight_decay=1e-5,
        batch_size=64,
        epochs=100,
        patience=10,
        loss_threshold=1e-4,
        num_buildings=None
    ):
        self.return_sequence = return_sequence
        self.latent_dim = latent_dim
        self.enc_hidden = enc_hidden
        self.dec_hidden = dec_hidden
        self.enc_out_hidden = enc_out_hidden
        self.dec_out_hidden = dec_out_hidden
        self.enc_layers = enc_layers
        self.dec_layers = dec_layers
        self.dropout = dropout
        self.lr = lr
        self.weight_decay = weight_decay
        self.batch_size = batch_size
        self.epochs = epochs
        self.patience = patience
        self.loss_threshold = loss_threshold
        self.num_buildings = num_buildings

        self.model_ = None
        self.history_ = None


    def fit(self, X_train, y_train, b_train, X_valid=None, y_valid=None, b_valid=None):
        X_train = np.asarray(X_train, dtype=np.float32)
        y_train = np.asarray(y_train, dtype=np.float32)
        if y_train.ndim == 1:
            y_train = y_train[:, None]

        if X_valid is None:
            X_valid = X_train
            y_valid = y_train
            b_valid = b_train
        else:
            X_valid = np.asarray(X_valid, dtype=np.float32)
            y_valid = np.asarray(y_valid, dtype=np.float32)
            if y_valid.ndim == 1:
                y_valid = y_valid[:, None]

        train_loader = DataLoader(
            PrebuiltSequenceDataset(X_train, y_train, b_train),
            batch_size=self.batch_size,num_workers=0,   # or 4
            shuffle=True
        )
        valid_loader = DataLoader(
            PrebuiltSequenceDataset(X_valid, y_valid, b_valid),
            batch_size=self.batch_size,num_workers=0,   # or 4
            shuffle=False
        )

        self.model_ = LatentStateEnergyModel(
            obs_dim=X_train.shape[-1],
            num_buildings=self.num_buildings,
            latent_dim=self.latent_dim,
            enc_hidden=self.enc_hidden,
            dec_hidden=self.dec_hidden,
            enc_out_hidden=self.enc_out_hidden,
            dec_out_hidden=self.dec_out_hidden,
            enc_layers=self.enc_layers,
            dec_layers=self.dec_layers,
            dropout=self.dropout,
            y_dim=1
        ).to(device)


        optimizer = torch.optim.Adam(
            self.model_.parameters(),
            lr=self.lr,
            weight_decay=self.weight_decay
        )
        scheduler = torch.optim.lr_scheduler.StepLR(
            optimizer,
            step_size=schedulerstep,
            gamma=0.75
        )

        best_val = float("inf")
        best_state = None
        patience_counter = 0
        self.history_ = {"train_loss": [], "valid_loss": []}

        for epoch in range(1, self.epochs + 1):
            train_loss = train_one_epoch_em(self.model_, train_loader, optimizer)
            valid_loss = valid_one_epoch_em(self.model_, valid_loader)
            val_rmse,val_rmselog = compute_rmse(self, X_valid, y_valid, b_valid)

            self.history_["train_loss"].append(train_loss)
            self.history_["valid_loss"].append(valid_loss)

            print(f"Epoch {epoch:03d} | "
                f"train loss={train_loss:.6f} | "
                f"valid loss={valid_loss:.6f} | "
                f"val_LogRMSE={val_rmselog:.6f} | ")
            scheduler.step()
            if train_loss < best_val - self.loss_threshold:
                best_val = train_loss
                best_state = copy.deepcopy(self.model_.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1
                print("Early stopping",patience_counter)
                if patience_counter >= self.patience:
                    print("Early stopping")
                    break

        if best_state is not None:
            self.model_.load_state_dict(best_state)

        return self

    def predict(self, X, building_ids):
      self.model_.eval()

      X = np.asarray(X, dtype=np.float32)
      building_ids = np.asarray(building_ids, dtype=np.int64)
      dummy_y = np.zeros((X.shape[0], 1), dtype=np.float32)

      loader = DataLoader(
          PrebuiltSequenceDataset(X, dummy_y, building_ids),
          batch_size=self.batch_size,
          shuffle=False
      )

      all_preds = []

      with torch.no_grad():
          for obs, _, b_ids in loader:
              obs = obs.to(device)
              b_ids = b_ids.to(device)

              x0, x_mu, x_logvar = self.model_.encode(obs, b_ids)
              y_mu, _ = self.model_.decode_from_x(obs, x0, x_mu)

              all_preds.append(y_mu.cpu().numpy())

      pred_mean = np.concatenate(all_preds, axis=0).reshape(-1)
      return pred_mean

    def predict_dist(self, X, building_ids):
        self.model_.eval()

        X = np.asarray(X, dtype=np.float32)
        building_ids = np.asarray(building_ids, dtype=np.int64)
        dummy_y = np.zeros((X.shape[0], 1), dtype=np.float32)

        loader = DataLoader(
            PrebuiltSequenceDataset(X, dummy_y, building_ids),
            batch_size=self.batch_size,num_workers=0,   # or 4
            shuffle=False
        )

        all_mean = []
        all_std = []

        with torch.no_grad():
            for obs, _, b_ids in loader:
                obs = obs.to(device)
                b_ids = b_ids.to(device)

                mu, var = mc_predict_batch(self.model_, obs, b_ids, mc_samples=MC_SAMPLES)
                std = torch.sqrt(var)

                all_mean.append(mu.cpu().numpy())   # [B, 1]
                all_std.append(std.cpu().numpy())   # [B, 1]

        pred_mean = np.concatenate(all_mean, axis=0).reshape(-1)
        pred_std = np.concatenate(all_std, axis=0).reshape(-1)
        return pred_mean, pred_std



# Model Parameters and Hyperparameters

In [ ]:
# =========================================================
# Hyperparameters
# =========================================================
# Reproducibility / Device
SEED = 42
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
set_seed(SEED)
horizon = 0


# Training Parameters
EPOCHS = 100   # maximum epochs
PATIENCE = 5
WEIGHT_DECAY = 1e-5
Optimzier = 'Adam'
Loss_threshold = 0.01
Lambda_L2=0.001
schedulerstep=100 # optional, not used
# Model Architecture

# # Model Hyperparameters
LATENT_DIM = 2
ENC_HIDDEN = 128  # LSTM Encoder
DEC_HIDDEN = 64  # LSTM Decoder
Enc_Out_Hidden=64 # decoder middel layer
Dec_Out_Hidden=64 # encoder middle layer
ENC_LAYERS = 2
DEC_LAYERS = 1
DROPOUT = 0.01
N_PARTICLES = 5
MC_SAMPLES = 1
seq_len=12
LR=0.005
BATCH_SIZE=128




Device: cuda


# Transfer Learning (Main)

In [ ]:

# =========================================================
# Config
# =========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

target_building_id = 672

schedulerstep = 1000

# split target building chronologically
target_tune_ratio = 0.08
target_test_ratio = 0.50

# use same source amount as pretraining
source_sequence_ratio = 0.1

# training configs
pretrain_epochs = 50
pretrain_patience = 10

finetune_epochs = 50
finetune_patience = 10

BATCH_SIZE_Finetune = 32
finetune_lr = 1e-4

scratch_epochs = 50
scratch_patience = 10

lambda_reg = 1e-3

# optional saving
SAVE_RESULTS = True
RESULTS_PATH = "transfer_benchmark_results.csv"

# =========================================================
# Helpers
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def subset_sequences(X, y, b, frac, seed=42):
    n = len(X)
    if n == 0:
        return X, y, b
    take = max(1, int(n * frac))
    rng = np.random.default_rng(seed)
    idx = rng.choice(n, size=take, replace=False)
    return X[idx], y[idx], b[idx]

def chronological_split_target(df, time_col, tune_ratio=0.495, test_ratio=0.50):
    df = df.sort_values(time_col).reset_index(drop=True)
    n = len(df)

    n_tune = int(n * tune_ratio)
    n_test = int(n * test_ratio)

    tune_df = df.iloc[:n_tune].copy()
    test_df = df.iloc[n - n_test:].copy()

    middle_start = n_tune
    middle_end = n - n_test
    gap_df = df.iloc[middle_start:middle_end].copy()

    return tune_df, gap_df, test_df

def freeze_encoder_train_decoder(model):
    # freeze everything first
    for p in model.parameters():
        p.requires_grad = False

    # freeze encoder + encoder heads
    for p in model.encoder.parameters():
        p.requires_grad = False
    for p in model.enc_mu.parameters():
        p.requires_grad = False
    for p in model.enc_logvar.parameters():
        p.requires_grad = False

    # keep decoder + decoder heads trainable
    for p in model.decoder.parameters():
        p.requires_grad = True
    for p in model.dec_mu.parameters():
        p.requires_grad = True
    for p in model.dec_logvar.parameters():
        p.requires_grad = True

    # keep building embedding trainable
    for p in model.x0_net.parameters():
        p.requires_grad = True

def decoder_l2_sp_regularization(model, decoder_ref_state, lambda_reg=1e-4):
    reg = torch.tensor(0.0, device=next(model.parameters()).device)
    named_params = dict(model.named_parameters())

    for name, ref_tensor in decoder_ref_state.items():
        if name in named_params:
            reg = reg + torch.sum((named_params[name] - ref_tensor) ** 2)

    return lambda_reg * reg

def evaluate_all_metrics(model_wrapper, X_test, y_test, b_test, model_name="model"):
    """
    Returns a dict with all metrics + predictions.
    Assumes Model_Evaluate returns:
    y_true, y_pred, log_rmse, rmse, r2, mad, mape, wape
    """
    y_true, y_pred, log_rmse, rmse, r2, mad, mape, wape = Model_Evaluate(
        model_wrapper,
        X_test,
        y_test,
        b_test,
        model_name=model_name,
        benchmark="no"
    )

    # compute log-rmse explicitly to avoid return-order confusion
    y_true_arr = np.asarray(y_true).reshape(-1)
    y_pred_arr = np.asarray(y_pred).reshape(-1)

    y_true_log = np.log1p(np.clip(y_true_arr, a_min=0, a_max=None))
    y_pred_log = np.log1p(np.clip(y_pred_arr, a_min=0, a_max=None))
    log_rmse = np.sqrt(np.mean((y_true_log - y_pred_log) ** 2))

    return {
        "rmse": float(rmse),
        "log_rmse": float(log_rmse),
        "r2": float(r2),
        "mad": float(mad),
        "mape": float(mape),
        "wape": float(wape),
        "y_true": y_true_arr,
        "y_pred": y_pred_arr,
    }

# =========================================================
# Training wrappers
# =========================================================
class EMTransferWrapper:
    """
    Light wrapper so we can use Model_Evaluate(model_wrapper, ...)
    the same way for pretrained / finetuned raw torch models.
    """
    def __init__(self, model):
        self.model_ = model

    def predict(self, X, building_ids):
        self.model_.eval()

        X = np.asarray(X, dtype=np.float32)
        building_ids = np.asarray(building_ids, dtype=np.int64)
        dummy_y = np.zeros((X.shape[0], 1), dtype=np.float32)

        loader = DataLoader(
            PrebuiltSequenceDataset(X, dummy_y, building_ids),
            batch_size=BATCH_SIZE_Finetune,
            shuffle=False
        )

        all_preds = []

        with torch.no_grad():
            for obs, _, b_ids in loader:
                obs = obs.to(device)
                b_ids = b_ids.to(device)

                x0, x_mu, x_logvar = self.model_.encode(obs, b_ids)
                y_mu, _ = self.model_.decode_from_x(obs, x0, x_mu)
                all_preds.append(y_mu.cpu().numpy())

        pred_mean = np.concatenate(all_preds, axis=0).reshape(-1)
        print(pred_mean)
        return pred_mean

def build_em_model(
    obs_dim,
    num_buildings,
    latent_dim=LATENT_DIM,
    enc_hidden=ENC_HIDDEN,
    dec_hidden=DEC_HIDDEN,
    enc_out_hidden=Enc_Out_Hidden,
    dec_out_hidden=Dec_Out_Hidden,
    enc_layers=ENC_LAYERS,
    dec_layers=DEC_LAYERS,
    dropout=DROPOUT,
    y_dim=1
):
    model = LatentStateEnergyModel(
        obs_dim=obs_dim,
        num_buildings=num_buildings,
        latent_dim=latent_dim,
        enc_hidden=enc_hidden,
        dec_hidden=dec_hidden,
        enc_out_hidden=enc_out_hidden,
        dec_out_hidden=dec_out_hidden,
        enc_layers=enc_layers,
        dec_layers=dec_layers,
        dropout=dropout,
        y_dim=y_dim
    ).to(device)
    return model

def train_raw_model_no_validation(
    model,
    X_train,
    y_train,
    b_train,
    lr,
    weight_decay,
    batch_size,
    epochs,
    patience,
    verbose=True
):
    X_train = np.asarray(X_train, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.float32)
    b_train = np.asarray(b_train, dtype=np.int64)

    if y_train.ndim == 1:
        y_train = y_train[:, None]

    train_loader = DataLoader(
        PrebuiltSequenceDataset(X_train, y_train, b_train),
        batch_size=batch_size,
        shuffle=True
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    best_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    patience_counter = 0
    history = []

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch_em(model, train_loader, optimizer)
        history.append(train_loss)

        if verbose:
            print(f"[Raw Train] epoch={epoch:03d} train_loss={train_loss:.6f}")

        if train_loss < best_loss - Loss_threshold:
            best_loss = train_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                if verbose:
                    print("[Raw Train] Early stopping")
                break

    model.load_state_dict(best_state)
    return model, history

def finetune_decoder_only(
    model,
    X_train,
    y_train,
    b_train,
    lr,
    weight_decay,
    batch_size,
    epochs,
    patience,
    lambda_reg=1e-4,
    verbose=True
):
    X_train = np.asarray(X_train, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.float32)
    b_train = np.asarray(b_train, dtype=np.int64)

    if y_train.ndim == 1:
        y_train = y_train[:, None]

    train_loader = DataLoader(
        PrebuiltSequenceDataset(X_train, y_train, b_train),
        batch_size=batch_size,
        shuffle=True
    )

    freeze_encoder_train_decoder(model)
    model.train()
    model.encoder.eval()

    if verbose:
        print("\n[Debug] Trainable parameters after freezing:")
        for name, p in model.named_parameters():
            if p.requires_grad:
                print(f"  {name} {tuple(p.shape)}")

    # Exclude x0_net from L2-SP so unseen target building embedding can adapt
    decoder_ref_state = {
        name: p.detach().clone()
        for name, p in model.named_parameters()
        if p.requires_grad and not name.startswith("x0_net")
    }

    trainable_named_params = [
        (name, p) for name, p in model.named_parameters()
        if p.requires_grad
    ]
    trainable_params = [p for _, p in trainable_named_params]

    optimizer = torch.optim.Adam(
        trainable_params,
        lr=lr,
        weight_decay=weight_decay
    )

    best_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    patience_counter = 0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()

        epoch_y_loss = 0.0
        epoch_x_loss = 0.0
        epoch_reg_raw = 0.0
        epoch_total_loss = 0.0
        n_batches = 0

        for obs, y_true, b_ids in train_loader:
            obs = obs.to(device)
            y_true = y_true.to(device)
            b_ids = b_ids.to(device)

            if y_true.ndim == 1:
                y_true = y_true.unsqueeze(-1)

            optimizer.zero_grad()

            # ===== SAME EM FLOW AS ORIGINAL TRAINING =====
            model.eval()
            with torch.no_grad():
                resampled_x = e_step_sample_and_weight(
                    model, obs, y_true, b_ids,
                    n_particles=N_PARTICLES
                )

            model.train()
            loss_em, y_loss, x_loss = m_step_loss(
                model, obs, y_true, b_ids, resampled_x
            )

            reg_raw = torch.tensor(0.0, device=device)
            for name, p in trainable_named_params:
                if name in decoder_ref_state:
                    reg_raw = reg_raw + torch.sum((p - decoder_ref_state[name]) ** 2)

            total_loss = loss_em + lambda_reg * reg_raw
            total_loss.backward()

            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            optimizer.step()

            if verbose and epoch == 1 and n_batches == 0:
                print("\n[Debug] Gradient norms on first batch:")
                for name, p in trainable_named_params:
                    grad_norm = None if p.grad is None else p.grad.norm().item()
                    print(f"  {name}: {grad_norm}")

            epoch_y_loss += y_loss.item()
            epoch_x_loss += x_loss.item()
            epoch_reg_raw += reg_raw.item()
            epoch_total_loss += total_loss.item()
            n_batches += 1

        avg_y_loss = epoch_y_loss / max(n_batches, 1)
        avg_x_loss = epoch_x_loss / max(n_batches, 1)
        avg_reg_raw = epoch_reg_raw / max(n_batches, 1)
        avg_total_loss = epoch_total_loss / max(n_batches, 1)

        history.append({
            "epoch": epoch,
            "y_loss": avg_y_loss,
            "x_loss": avg_x_loss,
            "reg_raw": avg_reg_raw,
            "reg_weighted": lambda_reg * avg_reg_raw,
            "total_loss": avg_total_loss,
        })

        if verbose:
            print(
                f"[Finetune-EM] epoch={epoch:03d} "
                f"y_loss={avg_y_loss:.6f} "
                f"x_loss={avg_x_loss:.6f} "
                f"reg_raw={avg_reg_raw:.6f} "
                f"reg_weighted={(lambda_reg * avg_reg_raw):.6f} "
                f"total_loss={avg_total_loss:.6f}"
            )

        if avg_total_loss < best_loss - Loss_threshold:
            best_loss = avg_total_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                if verbose:
                    print("[Finetune-EM] Early stopping")
                break

    model.load_state_dict(best_state)
    return model, history

# =========================================================
# Prepare data
# =========================================================
df_all = train_featuresNormalized.copy()

target_df = df_all[df_all[building_col] == target_building_id].copy()
source_df = df_all[df_all[building_col] != target_building_id].copy()

target_tune_df, target_gap_df, target_test_df = chronological_split_target(
    target_df,
    time_col="timestamp",
    tune_ratio=target_tune_ratio,
    test_ratio=target_test_ratio
)

print("Target building:", target_building_id)
print("Source rows:", len(source_df))
print("Target tune rows:", len(target_tune_df))
print("Target gap rows:", len(target_gap_df))
print("Target test rows:", len(target_test_df))

# =========================================================
# Build sequences
# =========================================================
X_source_full, y_source_full, b_source_full = make_sequences_by_building(
    source_df,
    feature_cols,
    target_col,
    building_col,
    building_idx_col="building_idx",
    seq_len=seq_len,
    horizon=horizon
)

X_tune, y_tune, b_tune = make_sequences_by_building(
    target_tune_df,
    feature_cols,
    target_col,
    building_col,
    building_idx_col="building_idx",
    seq_len=seq_len,
    horizon=horizon
)

X_test, y_test, b_test = make_sequences_by_building(
    target_test_df,
    feature_cols,
    target_col,
    building_col,
    building_idx_col="building_idx",
    seq_len=seq_len,
    horizon=horizon
)

if len(X_tune) == 0 or len(X_test) == 0 or len(X_source_full) == 0:
    raise ValueError("One split has zero sequences. Check target split sizes and sequence generation.")

X_source, y_source, b_source = subset_sequences(
    X_source_full,
    y_source_full,
    b_source_full,
    frac=source_sequence_ratio,
    seed=SEED
)

print("Source sequences used:", len(X_source))
print("Target tune sequences:", len(X_tune))
print("Target test sequences:", len(X_test))

# Naive aggregation train set = same source amount + target tune
X_naive = np.concatenate([X_source, X_tune], axis=0)
y_naive = np.concatenate([y_source, y_tune], axis=0)
b_naive = np.concatenate([b_source, b_tune], axis=0)

# =========================================================
# Experiment 1: Transfer Learning
# =========================================================
set_seed(SEED)

transfer_model = build_em_model(
    obs_dim=X_source.shape[-1],
    num_buildings=num_buildings
)

print("\n================ TRANSFER: PRETRAIN ================")
transfer_model, pretrain_history = train_raw_model_no_validation(
    model=transfer_model,
    X_train=X_source,
    y_train=y_source,
    b_train=b_source,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    batch_size=BATCH_SIZE,
    epochs=pretrain_epochs,
    patience=pretrain_patience,
    verbose=True
)

print("\n================ TRANSFER: FINETUNE ================")
transfer_model_tuned, finetune_history = finetune_decoder_only(
    model=transfer_model,
    X_train=X_tune,
    y_train=y_tune,
    b_train=b_tune,
    lr=finetune_lr,
    weight_decay=WEIGHT_DECAY,
    batch_size=BATCH_SIZE_Finetune,
    epochs=finetune_epochs,
    patience=finetune_patience,
    lambda_reg=lambda_reg,
    verbose=True
)

transfer_wrapper = EMTransferWrapper(transfer_model_tuned)
transfer_metrics = evaluate_all_metrics(
    transfer_wrapper,
    X_test,
    y_test,
    b_test,
    model_name="transfer_learning"
)

# =========================================================
# Experiment 2: Naive Aggregation
# =========================================================
set_seed(SEED)

naive_model = build_em_model(
    obs_dim=X_naive.shape[-1],
    num_buildings=num_buildings
)

print("\n================ NAIVE AGGREGATION ================")
naive_model, naive_history = train_raw_model_no_validation(
    model=naive_model,
    X_train=X_naive,
    y_train=y_naive,
    b_train=b_naive,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    batch_size=BATCH_SIZE,
    epochs=scratch_epochs,
    patience=scratch_patience,
    verbose=True
)

naive_wrapper = EMTransferWrapper(naive_model)
naive_metrics = evaluate_all_metrics(
    naive_wrapper,
    X_test,
    y_test,
    b_test,
    model_name="naive_aggregation"
)

# =========================================================
# Experiment 3: Target Only
# =========================================================
set_seed(SEED)

target_only_model = EMSeq2SeqRegressor(
    return_sequence=False,
    latent_dim=LATENT_DIM,
    enc_hidden=ENC_HIDDEN,
    dec_hidden=DEC_HIDDEN,
    enc_out_hidden=Enc_Out_Hidden,
    dec_out_hidden=Dec_Out_Hidden,
    enc_layers=ENC_LAYERS,
    dec_layers=DEC_LAYERS,
    dropout=DROPOUT,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    batch_size=BATCH_SIZE,
    epochs=scratch_epochs,
    patience=scratch_patience,
    loss_threshold=Loss_threshold,
    num_buildings=num_buildings
)

print("\n================ TARGET ONLY ================")
target_only_model.fit(X_tune, y_tune, b_tune, X_test, y_test, b_test)

target_only_metrics = evaluate_all_metrics(
    target_only_model,
    X_test,
    y_test,
    b_test,
    model_name="target_only"
)

# =========================================================
# Collect results
# =========================================================
results_rows = [
    {
        "method": "transfer_learning",
        "target_building_id": target_building_id,
        "source_sequences_used": len(X_source),
        "target_tune_sequences": len(X_tune),
        "target_test_sequences": len(X_test),
        "rmse": transfer_metrics["rmse"],
        "log_rmse": transfer_metrics["log_rmse"],
        "r2": transfer_metrics["r2"],
        "mad": transfer_metrics["mad"],
        "mape": transfer_metrics["mape"],
        "wape": transfer_metrics["wape"],
    },
    {
        "method": "naive_aggregation",
        "target_building_id": target_building_id,
        "source_sequences_used": len(X_source),
        "target_tune_sequences": len(X_tune),
        "target_test_sequences": len(X_test),
        "rmse": naive_metrics["rmse"],
        "log_rmse": naive_metrics["log_rmse"],
        "r2": naive_metrics["r2"],
        "mad": naive_metrics["mad"],
        "mape": naive_metrics["mape"],
        "wape": naive_metrics["wape"],
    },
    {
        "method": "target_only",
        "target_building_id": target_building_id,
        "source_sequences_used": 0,
        "target_tune_sequences": len(X_tune),
        "target_test_sequences": len(X_test),
        "rmse": target_only_metrics["rmse"],
        "log_rmse": target_only_metrics["log_rmse"],
        "r2": target_only_metrics["r2"],
        "mad": target_only_metrics["mad"],
        "mape": target_only_metrics["mape"],
        "wape": target_only_metrics["wape"],
    }
]

results_df = pd.DataFrame(results_rows).sort_values("log_rmse").reset_index(drop=True)

print("\n================ FINAL RESULTS ================")
print(results_df.to_string(index=False))

if SAVE_RESULTS:
    results_df.to_csv(RESULTS_PATH, index=False)
    print(f"\nSaved results to: {RESULTS_PATH}")

Target building: 672
Source rows: 179509
Target tune rows: 702
Target gap rows: 3690
Target test rows: 4392
Source sequences used: 17927
Target tune sequences: 691
Target test sequences: 4381

================ TRANSFER: PRETRAIN ================
[Raw Train] epoch=001 train_loss=1.793470
[Raw Train] epoch=002 train_loss=1.261977
[Raw Train] epoch=003 train_loss=0.698340
[Raw Train] epoch=004 train_loss=0.447301
[Raw Train] epoch=005 train_loss=0.349604
[Raw Train] epoch=006 train_loss=0.294023
[Raw Train] epoch=007 train_loss=0.265364
[Raw Train] epoch=008 train_loss=0.176475
[Raw Train] epoch=009 train_loss=0.153705
[Raw Train] epoch=010 train_loss=0.109892
[Raw Train] epoch=011 train_loss=0.078520
[Raw Train] epoch=012 train_loss=0.064341
[Raw Train] epoch=013 train_loss=0.043596
[Raw Train] epoch=014 train_loss=0.049009
[Raw Train] epoch=015 train_loss=0.071190
[Raw Train] epoch=016 train_loss=0.009377
[Raw Train] epoch=017 train_loss=-0.044717
[Raw Train] epoch=018 train_loss=-0.059

# Tranfer leaning with cross-validation (Leave-One-Out)

In [ ]:

# =========================================================
# Setup Parameters
# =========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

schedulerstep = 1000 # Not used

# split target building chronologically
target_tune_ratio = 0.04
target_test_ratio = 0.50

# use source amount as pretraining
source_sequence_ratio = 0.1

# training configs
pretrain_epochs = 50
pretrain_patience = 10

finetune_epochs = 100
finetune_patience = 10

BATCH_SIZE_Finetune = 16
finetune_lr = 1e-4

scratch_epochs = 50
scratch_patience = 10

lambda_reg = 1e-3

# optional saving
SAVE_RESULTS = True
RESULTS_PATH = "transfer_benchmark_results.csv"
SUMMARY_RESULTS_PATH = "transfer_benchmark_summary.csv"
WINS_RESULTS_PATH = "transfer_benchmark_wins.csv"

# =========================================================
# Helpers
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def subset_sequences(X, y, b, frac, seed=42):
    n = len(X)
    if n == 0:
        return X, y, b
    take = max(1, int(n * frac))
    rng = np.random.default_rng(seed)
    idx = rng.choice(n, size=take, replace=False)
    return X[idx], y[idx], b[idx]

def chronological_split_target(df, time_col, tune_ratio=0.495, test_ratio=0.50):
    df = df.sort_values(time_col).reset_index(drop=True)
    n = len(df)

    n_tune = int(n * tune_ratio)
    n_test = int(n * test_ratio)

    tune_df = df.iloc[:n_tune].copy()
    test_df = df.iloc[n - n_test:].copy()

    middle_start = n_tune
    middle_end = n - n_test
    gap_df = df.iloc[middle_start:middle_end].copy()

    return tune_df, gap_df, test_df

def freeze_encoder_train_decoder(model):
    # freeze everything first
    for p in model.parameters():
        p.requires_grad = False

    # keep encoder frozen
    for p in model.encoder.parameters():
        p.requires_grad = False
    for p in model.enc_mu.parameters():
        p.requires_grad = False
    for p in model.enc_logvar.parameters():
        p.requires_grad = False

    # unfreeze decoder + decoder heads
    for p in model.decoder.parameters():
        p.requires_grad = True
    for p in model.dec_mu.parameters():
        p.requires_grad = True
    for p in model.dec_logvar.parameters():
        p.requires_grad = True

    # keep building embedding trainable
    for p in model.x0_net.parameters():
        p.requires_grad = True

def decoder_l2_sp_regularization(model, decoder_ref_state, lambda_reg=1e-4):
    reg = torch.tensor(0.0, device=next(model.parameters()).device)
    named_params = dict(model.named_parameters())

    for name, ref_tensor in decoder_ref_state.items():
        if name in named_params:
            reg = reg + torch.sum((named_params[name] - ref_tensor) ** 2)

    return lambda_reg * reg

def evaluate_all_metrics(model_wrapper, X_test, y_test, b_test, model_name="model"):
    """
    Returns a dict with all metrics + predictions.
    Assumes Model_Evaluate returns:
    y_true, y_pred, log_rmse, rmse, r2, mad, mape, wape
    """

    y_true, y_pred, log_rmse, rmse, r2, mad, mape, wape = Model_Evaluate(
        model_wrapper,
        X_test,
        y_test,
        b_test,
        model_name=model_name,
        benchmark="no"
    )

    y_true_arr = np.asarray(y_true).reshape(-1)
    y_pred_arr = np.asarray(y_pred).reshape(-1)

    y_true_log = np.log1p(np.clip(y_true_arr, a_min=0, a_max=None))
    y_pred_log = np.log1p(np.clip(y_pred_arr, a_min=0, a_max=None))
    log_rmse = np.sqrt(np.mean((y_true_log - y_pred_log) ** 2))

    return {
        "rmse": float(rmse),
        "log_rmse": float(log_rmse),
        "r2": float(r2),
        "mad": float(mad),
        "mape": float(mape),
        "wape": float(wape),
        "y_true": y_true_arr,
        "y_pred": y_pred_arr,
    }

# =========================================================
# Training wrappers
# =========================================================
class EMTransferWrapper:
    """
    Light wrapper so we can use Model_Evaluate(model_wrapper, ...)
    the same way for pretrained / finetuned raw torch models.
    """
    def __init__(self, model):
        self.model_ = model

    def predict(self, X, building_ids):
        self.model_.eval()

        X = np.asarray(X, dtype=np.float32)
        building_ids = np.asarray(building_ids, dtype=np.int64)
        dummy_y = np.zeros((X.shape[0], 1), dtype=np.float32)

        loader = DataLoader(
            PrebuiltSequenceDataset(X, dummy_y, building_ids),
            batch_size=BATCH_SIZE_Finetune,
            shuffle=False
        )

        all_preds = []

        with torch.no_grad():
            for obs, _, b_ids in loader:
                obs = obs.to(device)
                b_ids = b_ids.to(device)

                x0, x_mu, x_logvar = self.model_.encode(obs, b_ids)
                y_mu, _ = self.model_.decode_from_x(obs, x0, x_mu)
                all_preds.append(y_mu.cpu().numpy())

        pred_mean = np.concatenate(all_preds, axis=0).reshape(-1)
        return pred_mean

def build_em_model(
    obs_dim,
    num_buildings,
    latent_dim=LATENT_DIM,
    enc_hidden=ENC_HIDDEN,
    dec_hidden=DEC_HIDDEN,
    enc_out_hidden=Enc_Out_Hidden,
    dec_out_hidden=Dec_Out_Hidden,
    enc_layers=ENC_LAYERS,
    dec_layers=DEC_LAYERS,
    dropout=DROPOUT,
    y_dim=1
):
    model = LatentStateEnergyModel(
        obs_dim=obs_dim,
        num_buildings=num_buildings,
        latent_dim=latent_dim,
        enc_hidden=enc_hidden,
        dec_hidden=dec_hidden,
        enc_out_hidden=enc_out_hidden,
        dec_out_hidden=dec_out_hidden,
        enc_layers=enc_layers,
        dec_layers=dec_layers,
        dropout=dropout,
        y_dim=y_dim
    ).to(device)
    return model

def train_raw_model_no_validation(
    model,
    X_train,
    y_train,
    b_train,
    lr,
    weight_decay,
    batch_size,
    epochs,
    patience,
    verbose=True
):
    X_train = np.asarray(X_train, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.float32)
    b_train = np.asarray(b_train, dtype=np.int64)

    if y_train.ndim == 1:
        y_train = y_train[:, None]

    train_loader = DataLoader(
        PrebuiltSequenceDataset(X_train, y_train, b_train),
        batch_size=batch_size,
        shuffle=True
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    best_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    patience_counter = 0
    history = []

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch_em(model, train_loader, optimizer)
        history.append(train_loss)

        if verbose:
            print(f"[Raw Train] epoch={epoch:03d} train_loss={train_loss:.6f}")

        if train_loss < best_loss - Loss_threshold:
            best_loss = train_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                if verbose:
                    print("[Raw Train] Early stopping")
                break

    model.load_state_dict(best_state)
    return model, history

def finetune_decoder_only(
    model,
    X_train,
    y_train,
    b_train,
    lr,
    weight_decay,
    batch_size,
    epochs,
    patience,
    lambda_reg=1e-4,
    verbose=True
):
    X_train = np.asarray(X_train, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.float32)
    b_train = np.asarray(b_train, dtype=np.int64)

    if y_train.ndim == 1:
        y_train = y_train[:, None]

    train_loader = DataLoader(
        PrebuiltSequenceDataset(X_train, y_train, b_train),
        batch_size=batch_size,
        shuffle=True
    )

    freeze_encoder_train_decoder(model)
    model.train()
    model.encoder.eval()

    if verbose:
        print("\n[Debug] Trainable parameters after freezing:")
        for name, p in model.named_parameters():
            if p.requires_grad:
                print(f"  {name} {tuple(p.shape)}")

    # Exclude x0_net from L2-SP so target building embedding can adapt
    decoder_ref_state = {
        name: p.detach().clone()
        for name, p in model.named_parameters()
        if p.requires_grad and not name.startswith("x0_net")
    }

    trainable_named_params = [
        (name, p) for name, p in model.named_parameters()
        if p.requires_grad
    ]
    trainable_params = [p for _, p in trainable_named_params]

    optimizer = torch.optim.Adam(
        trainable_params,
        lr=lr,
        weight_decay=weight_decay
    )

    best_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    patience_counter = 0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()

        epoch_y_loss = 0.0
        epoch_x_loss = 0.0
        epoch_reg_raw = 0.0
        epoch_total_loss = 0.0
        n_batches = 0

        for obs, y_true, b_ids in train_loader:
            obs = obs.to(device)
            y_true = y_true.to(device)
            b_ids = b_ids.to(device)

            if y_true.ndim == 1:
                y_true = y_true.unsqueeze(-1)

            optimizer.zero_grad()

            # same EM flow as original training
            model.eval()
            with torch.no_grad():
                resampled_x = e_step_sample_and_weight(
                    model, obs, y_true, b_ids,
                    n_particles=N_PARTICLES
                )

            model.train()
            loss_em, y_loss, x_loss = m_step_loss(
                model, obs, y_true, b_ids, resampled_x
            )

            reg_raw = torch.tensor(0.0, device=device)
            for name, p in trainable_named_params:
                if name in decoder_ref_state:
                    reg_raw = reg_raw + torch.sum((p - decoder_ref_state[name]) ** 2)

            total_loss = loss_em + lambda_reg * reg_raw
            total_loss.backward()

            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            optimizer.step()

            if verbose and epoch == 1 and n_batches == 0:
                print("\n[Debug] Gradient norms on first batch:")
                for name, p in trainable_named_params:
                    grad_norm = None if p.grad is None else p.grad.norm().item()
                    print(f"  {name}: {grad_norm}")

            epoch_y_loss += y_loss.item()
            epoch_x_loss += x_loss.item()
            epoch_reg_raw += reg_raw.item()
            epoch_total_loss += total_loss.item()
            n_batches += 1

        avg_y_loss = epoch_y_loss / max(n_batches, 1)
        avg_x_loss = epoch_x_loss / max(n_batches, 1)
        avg_reg_raw = epoch_reg_raw / max(n_batches, 1)
        avg_total_loss = epoch_total_loss / max(n_batches, 1)

        history.append({
            "epoch": epoch,
            "y_loss": avg_y_loss,
            "x_loss": avg_x_loss,
            "reg_raw": avg_reg_raw,
            "reg_weighted": lambda_reg * avg_reg_raw,
            "total_loss": avg_total_loss,
        })

        if verbose:
            print(
                f"[Finetune-EM] epoch={epoch:03d} "
                f"y_loss={avg_y_loss:.6f} "
                f"x_loss={avg_x_loss:.6f} "
                f"reg_raw={avg_reg_raw:.6f} "
                f"reg_weighted={(lambda_reg * avg_reg_raw):.6f} "
                f"total_loss={avg_total_loss:.6f}"
            )

        if avg_total_loss < best_loss - Loss_threshold:
            best_loss = avg_total_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                if verbose:
                    print("[Finetune-EM] Early stopping")
                break

    model.load_state_dict(best_state)
    return model, history

# =========================================================
# Prepare data
# =========================================================
df_all = train_featuresNormalized.copy()

all_target_building_ids = sorted(df_all[building_col].unique())

all_results_rows = []

metric_cols = ["rmse", "log_rmse", "r2", "mad", "mape", "wape"]
lower_is_better_metrics = ["rmse", "log_rmse", "mad", "mape", "wape"]
higher_is_better_metrics = ["r2"]

wins_counter = {
    "transfer_learning": {m: 0 for m in metric_cols},
    "naive_aggregation": {m: 0 for m in metric_cols},
    "target_only": {m: 0 for m in metric_cols},
}

for target_building_id in all_target_building_ids:
    print("\n" + "=" * 80)
    print(f"LEAVE-ONE-OUT TARGET BUILDING: {target_building_id}")
    print("=" * 80)

    target_df = df_all[df_all[building_col] == target_building_id].copy()
    source_df = df_all[df_all[building_col] != target_building_id].copy()

    target_tune_df, target_gap_df, target_test_df = chronological_split_target(
        target_df,
        time_col="timestamp",
        tune_ratio=target_tune_ratio,
        test_ratio=target_test_ratio
    )

    print("Target building:", target_building_id)
    print("Source rows:", len(source_df))
    print("Target tune rows:", len(target_tune_df))
    print("Target gap rows:", len(target_gap_df))
    print("Target test rows:", len(target_test_df))

    # =========================================================
    # Build sequences
    # =========================================================
    X_source_full, y_source_full, b_source_full = make_sequences_by_building(
        source_df,
        feature_cols,
        target_col,
        building_col,
        building_idx_col="building_idx",
        seq_len=seq_len,
        horizon=horizon
    )

    X_tune, y_tune, b_tune = make_sequences_by_building(
        target_tune_df,
        feature_cols,
        target_col,
        building_col,
        building_idx_col="building_idx",
        seq_len=seq_len,
        horizon=horizon
    )

    X_test, y_test, b_test = make_sequences_by_building(
        target_test_df,
        feature_cols,
        target_col,
        building_col,
        building_idx_col="building_idx",
        seq_len=seq_len,
        horizon=horizon
    )

    if len(X_tune) == 0 or len(X_test) == 0 or len(X_source_full) == 0:
        print(f"Skipping building {target_building_id} because one split has zero sequences.")
        continue

    X_source, y_source, b_source = subset_sequences(
        X_source_full,
        y_source_full,
        b_source_full,
        frac=source_sequence_ratio,
        seed=SEED
    )

    print("Source sequences used:", len(X_source))
    print("Target tune sequences:", len(X_tune))
    print("Target test sequences:", len(X_test))

    X_naive = np.concatenate([X_source, X_tune], axis=0)
    y_naive = np.concatenate([y_source, y_tune], axis=0)
    b_naive = np.concatenate([b_source, b_tune], axis=0)

    # =========================================================
    # Experiment 1: Transfer Learning
    # =========================================================
    set_seed(SEED)

    transfer_model = build_em_model(
        obs_dim=X_source.shape[-1],
        num_buildings=num_buildings
    )

    print("\n================ TRANSFER: PRETRAIN ================")
    transfer_model, pretrain_history = train_raw_model_no_validation(
        model=transfer_model,
        X_train=X_source,
        y_train=y_source,
        b_train=b_source,
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        batch_size=BATCH_SIZE,
        epochs=pretrain_epochs,
        patience=pretrain_patience,
        verbose=True
    )

    print("\n================ TRANSFER: FINETUNE ================")
    transfer_model, finetune_history = finetune_decoder_only(
        model=transfer_model,
        X_train=X_tune,
        y_train=y_tune,
        b_train=b_tune,
        lr=finetune_lr,
        weight_decay=WEIGHT_DECAY,
        batch_size=BATCH_SIZE_Finetune,
        epochs=finetune_epochs,
        patience=finetune_patience,
        lambda_reg=lambda_reg,
        verbose=True
    )

    transfer_wrapper = EMTransferWrapper(transfer_model)
    transfer_metrics = evaluate_all_metrics(
        transfer_wrapper,
        X_test,
        y_test,
        b_test,
        model_name="transfer_learning"
    )

    # =========================================================
    # Experiment 2: Naive Aggregation
    # =========================================================
    set_seed(SEED)

    print("\n================ NAIVE AGGREGATION ================")
    naive_model = build_em_model(
        obs_dim=X_naive.shape[-1],
        num_buildings=num_buildings
    )

    naive_model, naive_history = train_raw_model_no_validation(
        model=naive_model,
        X_train=X_naive,
        y_train=y_naive,
        b_train=b_naive,
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        batch_size=BATCH_SIZE,
        epochs=scratch_epochs,
        patience=scratch_patience,
        verbose=True
    )

    naive_wrapper = EMTransferWrapper(naive_model)
    naive_metrics = evaluate_all_metrics(
        naive_wrapper,
        X_test,
        y_test,
        b_test,
        model_name="naive_aggregation"
    )

    # =========================================================
    # Experiment 3: Target Only
    # =========================================================
    set_seed(SEED)

    target_only_model = EMSeq2SeqRegressor(
        return_sequence=False,
        latent_dim=LATENT_DIM,
        enc_hidden=ENC_HIDDEN,
        dec_hidden=DEC_HIDDEN,
        enc_out_hidden=Enc_Out_Hidden,
        dec_out_hidden=Dec_Out_Hidden,
        enc_layers=ENC_LAYERS,
        dec_layers=DEC_LAYERS,
        dropout=DROPOUT,
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        batch_size=BATCH_SIZE,
        epochs=scratch_epochs,
        patience=scratch_patience,
        loss_threshold=Loss_threshold,
        num_buildings=num_buildings
    )

    print("\n================ TARGET ONLY ================")
    target_only_model.fit(X_tune, y_tune, b_tune, X_test, y_test, b_test)

    target_only_metrics = evaluate_all_metrics(
        target_only_model,
        X_test,
        y_test,
        b_test,
        model_name="target_only"
    )

    # =========================================================
    # Collect results for this target building
    # =========================================================
    results_rows = [
        {
            "method": "transfer_learning",
            "target_building_id": target_building_id,
            "source_sequences_used": len(X_source),
            "target_tune_sequences": len(X_tune),
            "target_test_sequences": len(X_test),
            "rmse": transfer_metrics["rmse"],
            "log_rmse": transfer_metrics["log_rmse"],
            "r2": transfer_metrics["r2"],
            "mad": transfer_metrics["mad"],
            "mape": transfer_metrics["mape"],
            "wape": transfer_metrics["wape"],
        },
        {
            "method": "naive_aggregation",
            "target_building_id": target_building_id,
            "source_sequences_used": len(X_source),
            "target_tune_sequences": len(X_tune),
            "target_test_sequences": len(X_test),
            "rmse": naive_metrics["rmse"],
            "log_rmse": naive_metrics["log_rmse"],
            "r2": naive_metrics["r2"],
            "mad": naive_metrics["mad"],
            "mape": naive_metrics["mape"],
            "wape": naive_metrics["wape"],
        },
        {
            "method": "target_only",
            "target_building_id": target_building_id,
            "source_sequences_used": 0,
            "target_tune_sequences": len(X_tune),
            "target_test_sequences": len(X_test),
            "rmse": target_only_metrics["rmse"],
            "log_rmse": target_only_metrics["log_rmse"],
            "r2": target_only_metrics["r2"],
            "mad": target_only_metrics["mad"],
            "mape": target_only_metrics["mape"],
            "wape": target_only_metrics["wape"],
        }
    ]

    building_results_df = pd.DataFrame(results_rows)

    print("\n================ BUILDING RESULTS ================")
    print(building_results_df.sort_values("rmse").reset_index(drop=True).to_string(index=False))

    for metric in lower_is_better_metrics:
        best_value = building_results_df[metric].min()
        winners = building_results_df.loc[building_results_df[metric] == best_value, "method"].tolist()
        for winner in winners:
            wins_counter[winner][metric] += 1

    for metric in higher_is_better_metrics:
        best_value = building_results_df[metric].max()
        winners = building_results_df.loc[building_results_df[metric] == best_value, "method"].tolist()
        for winner in winners:
            wins_counter[winner][metric] += 1

    all_results_rows.extend(results_rows)

# =========================================================
# Final results across all buildings
# =========================================================
results_df = pd.DataFrame(all_results_rows)

print("\n================ ALL BUILDING RESULTS ================")
print(results_df.to_string(index=False))

summary_rows = []
for method_name, method_df in results_df.groupby("method"):
    row = {"method": method_name}
    for metric in metric_cols:
        row[f"{metric}_avg"] = method_df[metric].mean()
        row[f"{metric}_std"] = method_df[metric].std(ddof=1)
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

wins_rows = []
for method_name in wins_counter:
    row = {"method": method_name}
    for metric in metric_cols:
        row[f"{metric}_wins"] = wins_counter[method_name][metric]
    row["total_wins"] = sum(wins_counter[method_name][metric] for metric in metric_cols)
    wins_rows.append(row)

wins_df = pd.DataFrame(wins_rows)

print("\n================ AVERAGE AND STD ACROSS BUILDINGS ================")
print(summary_df.to_string(index=False))

print("\n================ MODEL WIN COUNTS ACROSS BUILDINGS ================")
print(wins_df.to_string(index=False))

if SAVE_RESULTS:
    results_df.to_csv(RESULTS_PATH, index=False)
    summary_df.to_csv(SUMMARY_RESULTS_PATH, index=False)
    wins_df.to_csv(WINS_RESULTS_PATH, index=False)

    print(f"\nSaved per-building results to: {RESULTS_PATH}")
    print(f"Saved summary results to: {SUMMARY_RESULTS_PATH}")
    print(f"Saved win counts to: {WINS_RESULTS_PATH}")

Streaming output truncated to the last 5000 lines.
[Finetune-EM] epoch=047 y_loss=-1.878074 x_loss=-0.081015 reg_raw=7.003985 reg_weighted=0.007004 total_loss=-1.950160
[Finetune-EM] epoch=048 y_loss=-1.895390 x_loss=-0.080070 reg_raw=7.076492 reg_weighted=0.007076 total_loss=-1.966458
[Finetune-EM] epoch=049 y_loss=-1.903159 x_loss=-0.046679 reg_raw=7.157632 reg_weighted=0.007158 total_loss=-1.940755
[Finetune-EM] epoch=050 y_loss=-1.910665 x_loss=-0.086206 reg_raw=7.241233 reg_weighted=0.007241 total_loss=-1.987706
[Finetune-EM] epoch=051 y_loss=-1.909508 x_loss=-0.073358 reg_raw=7.301230 reg_weighted=0.007301 total_loss=-1.973641
[Finetune-EM] Early stopping

Model: transfer_learning
Validation Log RMSE: 0.2269
Validation RMSE: 27.3990
Validation R²: -6.9244
Validation MAD: 22.0371
Validation MAPE: 18.4832
Validation WAPE: 21.7148%

================ NAIVE AGGREGATION ================
[Raw Train] epoch=001 train_loss=1.692838
[Raw Train] epoch=002 train_loss=1.349929
[Raw Train] epoc

# CV Results

In [ ]:
LATENT_DIM = 2 ENC_HIDDEN = 128  # LSTM Encoder DEC_HIDDEN = 64  # LSTM Decoder Enc_Out_Hidden=64
# decoder middel layer Dec_Out_Hidden=64 # encoder middle layer ENC_LAYERS = 2 DEC_LAYERS = 1
# DROPOUT = 0.01 N_PARTICLES = 5 MC_SAMPLES = 1 seq_len=12 LR=0.005 BATCH_SIZE=128


4% new
================ AVERAGE AND STD ACROSS BUILDINGS ================
           method  rmse_avg  rmse_std  log_rmse_avg  log_rmse_std    r2_avg    r2_std   mad_avg   mad_std  mape_avg  mape_std  wape_avg  wape_std
naive_aggregation 37.556216 34.351928      0.648685      0.378182 -3.165451  8.252481 28.789993 26.829737 46.495852 31.101636 52.385763 32.988029
      target_only 50.116666 45.687139      1.039927      0.484526 -3.614907  6.609346 38.961408 37.284857 50.297453 14.454414 53.894895 13.495386
transfer_learning 33.527163 30.610981      0.634380      0.365556 -3.028924 10.235144 25.383544 23.915459 37.568306 24.126684 43.461316 26.006064



16% new
================ AVERAGE AND STD ACROSS BUILDINGS ================
           method  rmse_avg  rmse_std  log_rmse_avg  log_rmse_std    r2_avg   r2_std   mad_avg   mad_std  mape_avg  mape_std  wape_avg  wape_std
naive_aggregation 36.380286 35.876770      0.615185      0.370325 -1.762313 4.716380 27.473101 27.430905 39.115833 22.562076 45.298111 24.750172
      target_only 46.215259 42.332627      0.949445      0.439315 -2.663007 3.787423 36.030767 33.895725 47.025363 16.276829 52.019513 19.063234
transfer_learning 34.847365 32.121434      0.617438      0.327804 -2.747026 7.229737 26.091642 24.967400 36.543178 20.395418 43.288011 22.534694




48% new

================ AVERAGE AND STD ACROSS BUILDINGS ================
           method  rmse_avg  rmse_std  log_rmse_avg  log_rmse_std    r2_avg   r2_std   mad_avg   mad_std  mape_avg  mape_std  wape_avg  wape_std
naive_aggregation 31.468026 35.595164      0.439297      0.187037 -0.780026 3.137003 22.845005 26.567601 23.729108 10.222032 31.181596 13.920392
      target_only 27.374441 26.146965      0.390343      0.167071  0.103424 0.394873 20.082562 20.293766 21.382303  8.068927 27.769518 12.821544
transfer_learning 24.901352 23.347568      0.392403      0.175764  0.166245 0.365605 17.943293 17.886874 21.316978  8.940098 26.850759 12.431264




0.005